In [62]:
import pandas as p
import warnings
warnings.filterwarnings("ignore")
import yfinance as yf

**Calculating volatility,drawdown and volume change and creating signals**

In [ ]:

all_stocks_data = p.read_csv("all_stocks_data.csv")

In [64]:
names = all_stocks_data["Name"].unique().tolist()

In [65]:
all_stocks_data = all_stocks_data.drop(columns=["Unnamed: 0","High","Low","Open"])

In [66]:
all_stocks_data

,Date,Close,Volume,Name
0,2024-01-01,2914.614990,2898619,Adani Enterprises Ltd.
1,2024-01-02,2929.801514,2671368,Adani Enterprises Ltd.
2,2024-01-03,3000.339111,19725411,Adani Enterprises Ltd.
3,2024-01-04,2995.643066,2975620,Adani Enterprises Ltd.
4,2024-01-05,3003.935791,3219949,Adani Enterprises Ltd.
...,...,...,...,...
26792,2026-03-09,198.750000,25862728,Wipro Ltd.
26793,2026-03-10,200.929993,12982340,Wipro Ltd.
26794,2026-03-11,202.229996,29199002,Wipro Ltd.
26795,2026-03-12,202.509995,19042475,Wipro Ltd.


In [67]:
all_stocks_data['Date'] = p.to_datetime(all_stocks_data['Date'])

In [68]:
all_stocks_data['quarter'] = all_stocks_data['Date'].dt.to_period('Q')

In [69]:
def calc_volatility(df):
  df['volatility'] = 0
  for i in range(1,len(df)-1):
   df['volatility'][i] = round(((df['Close'][i]/df['Close'][i-1])-1)*100,2)
  return df

In [70]:
def calc_drawdown(df):
  peak = df.groupby('quarter')['Close'].max().tolist()[0]
  df['drawdown'] = 0
  for i in range(len(df)):
    if df['Close'][i]>peak:
       peak = df['Close'][i]
       df['drawdown'][i] = 0
    else:
       df['drawdown'][i] = round((1-df['Close'][i]/peak)*100,2)
  return df

In [71]:
stock_analytics=p.DataFrame()

In [72]:
for stock in names:
    stock_df = all_stocks_data[all_stocks_data['Name'] == stock].reset_index(drop=True)
    print(stock)
    stock_df = calc_drawdown(calc_volatility(stock_df))
    stock_df = stock_df.groupby(['Name','quarter']).agg({'volatility':'std','drawdown':'max','Volume':'mean'}).reset_index()
    stock_df['volume_change'] = 0
    stock_df['drawdown_stress'] = ' '
    stock_df['volatility_signal'] = ' '
    stock_df['volume_signal'] = ' '
    for i in range(1,len(stock_df)):
        stock_df['volume_change'][i] = round((stock_df['Volume'][i]/stock_df['Volume'][i-1]-1)*100,2)
    stock_df['drawdown_stress'] = stock_df['drawdown'].apply(lambda x: 'High' if x>15 else ('Medium' if x>5 else 'Low'))
    stock_df['volume_signal'] = stock_df['volume_change'].apply(lambda x: 'Accumulation' if x>20 else ('Neutral' if x>-20 else 'Declining Interest'))
    for i in range(len(stock_df)):
     if stock_df['volatility'][i]>stock_df['volatility'].mean():
        stock_df['volatility_signal'][i] = 'Expansion'
     else:
        stock_df['volatility_signal'][i] = 'Compression'
    stock_analytics = p.concat([stock_analytics, stock_df])

Adani Enterprises Ltd.
Adani Ports and Special Economic Zone Ltd.
Apollo Hospitals Enterprise Ltd.
Asian Paints Ltd.
Axis Bank Ltd.
Bajaj Auto Ltd.
Bajaj Finance Ltd.
Bajaj Finserv Ltd.
Bharat Electronics Ltd.
Bharti Airtel Ltd.
Cipla Ltd.
Coal India Ltd.
Dr. Reddy's Laboratories Ltd.
Eicher Motors Ltd.
Eternal Ltd.
Grasim Industries Ltd.
HCL Technologies Ltd.
HDFC Bank Ltd.
HDFC Life Insurance Company Ltd.
Hindalco Industries Ltd.
Hindustan Unilever Ltd.
ICICI Bank Ltd.
ITC Ltd.
Infosys Ltd.
InterGlobe Aviation Ltd.
JSW Steel Ltd.
Jio Financial Services Ltd.
Kotak Mahindra Bank Ltd.
Larsen & Toubro Ltd.
Mahindra & Mahindra Ltd.
Maruti Suzuki India Ltd.
Max Healthcare Institute Ltd.
NTPC Ltd.
Nestle India Ltd.
Oil & Natural Gas Corporation Ltd.
Power Grid Corporation of India Ltd.
Reliance Industries Ltd.
SBI Life Insurance Company Ltd.
Shriram Finance Ltd.
State Bank of India
Sun Pharmaceutical Industries Ltd.
Tata Consultancy Services Ltd.
Tata Consumer Products Ltd.
Tata Motors Pass

In [73]:
stock_analytics = stock_analytics[['Name','quarter','drawdown_stress','volatility_signal','volume_signal']]

In [74]:
stock_analytics

,Name,quarter,drawdown_stress,volatility_signal,volume_signal
0,Adani Enterprises Ltd.,2024Q1,Medium,Compression,Neutral
1,Adani Enterprises Ltd.,2024Q2,High,Expansion,Neutral
2,Adani Enterprises Ltd.,2024Q3,High,Compression,Declining Interest
3,Adani Enterprises Ltd.,2024Q4,High,Expansion,Accumulation
4,Adani Enterprises Ltd.,2025Q1,High,Compression,Declining Interest
...,...,...,...,...,...
4,Wipro Ltd.,2025Q1,High,Expansion,Neutral
5,Wipro Ltd.,2025Q2,High,Expansion,Neutral
6,Wipro Ltd.,2025Q3,High,Compression,Declining Interest
7,Wipro Ltd.,2025Q4,High,Compression,Neutral


**Retrieving stock news**

In [75]:
updated_stocks = p.read_csv("D:/Stock Analyzer/updated_stocks.csv")

In [76]:
updated_stocks

,Company Name,Industry,Symbol,Series,ISIN Code
0,Adani Enterprises Ltd.,Metals & Mining,ADANIENT,EQ,INE423A01024
1,Adani Ports and Special Economic Zone Ltd.,Services,ADANIPORTS,EQ,INE742F01042
2,Apollo Hospitals Enterprise Ltd.,Healthcare,APOLLOHOSP,EQ,INE437A01024
3,Asian Paints Ltd.,Consumer Durables,ASIANPAINT,EQ,INE021A01026
4,Axis Bank Ltd.,Financial Services,AXISBANK,EQ,INE238A01034
5,Bajaj Auto Ltd.,Automobile and Auto Components,BAJAJ-AUTO,EQ,INE917I01010
6,Bajaj Finance Ltd.,Financial Services,BAJFINANCE,EQ,INE296A01032
7,Bajaj Finserv Ltd.,Financial Services,BAJAJFINSV,EQ,INE918I01026
8,Bharat Electronics Ltd.,Capital Goods,BEL,EQ,INE263A01024
9,Bharti Airtel Ltd.,Telecommunication,BHARTIARTL,EQ,INE397D01024


In [77]:
tickers = updated_stocks['Symbol'].tolist()

In [78]:
company_ticker_mapping = dict(zip(tickers, names))

In [79]:
stock_news_summary=[]
stock_news_name = []

In [80]:
for ticker in tickers:
 n = yf.Ticker(ticker+".NS")
 for i in range(len(n.news)):
    stock_news_summary.append(n.news[i]['content']['summary'])
    stock_news_name.append(company_ticker_mapping[ticker])
    print(ticker)
stock_news = p.DataFrame({'news':stock_news_summary,'name':stock_news_name})


ADANIENT
ADANIENT
ADANIENT
ADANIENT
ADANIENT
ADANIENT
ADANIENT
ADANIENT
ADANIENT
ADANIENT
ADANIPORTS
ADANIPORTS
ADANIPORTS
ADANIPORTS
ADANIPORTS
ADANIPORTS
ADANIPORTS
ADANIPORTS
ADANIPORTS
ADANIPORTS
APOLLOHOSP
APOLLOHOSP
APOLLOHOSP
APOLLOHOSP
APOLLOHOSP
APOLLOHOSP
APOLLOHOSP
APOLLOHOSP
APOLLOHOSP
APOLLOHOSP
ASIANPAINT
ASIANPAINT
ASIANPAINT
ASIANPAINT
ASIANPAINT
ASIANPAINT
ASIANPAINT
ASIANPAINT
ASIANPAINT
ASIANPAINT
AXISBANK
AXISBANK
AXISBANK
AXISBANK
AXISBANK
AXISBANK
AXISBANK
AXISBANK
AXISBANK
AXISBANK
BAJAJ-AUTO
BAJAJ-AUTO
BAJAJ-AUTO
BAJAJ-AUTO
BAJAJ-AUTO
BAJAJ-AUTO
BAJAJ-AUTO
BAJAJ-AUTO
BAJAJ-AUTO
BAJAJ-AUTO
BAJFINANCE
BAJFINANCE
BAJFINANCE
BAJFINANCE
BAJFINANCE
BAJFINANCE
BAJFINANCE
BAJFINANCE
BAJFINANCE
BAJFINANCE
BAJAJFINSV
BAJAJFINSV
BAJAJFINSV
BAJAJFINSV
BAJAJFINSV
BAJAJFINSV
BAJAJFINSV
BAJAJFINSV
BAJAJFINSV
BAJAJFINSV
BEL
BEL
BEL
BEL
BEL
BHARTIARTL
BHARTIARTL
BHARTIARTL
BHARTIARTL
BHARTIARTL
BHARTIARTL
BHARTIARTL
BHARTIARTL
BHARTIARTL
BHARTIARTL
CIPLA
CIPLA
CIPLA
CIPLA
CIPLA


In [81]:
stock_news

,news,name
0,"On Wednesday, the US imposed preliminary dutie...",Adani Enterprises Ltd.
1,Adani Enterprises says the move could catalyse...,Adani Enterprises Ltd.
2,Less than two weeks after New Delhi announced ...,Adani Enterprises Ltd.
3,The conglomerate’s investment would be a boost...,Adani Enterprises Ltd.
4,Indian billionaire Gautam Adani is joining the...,Adani Enterprises Ltd.
...,...,...
465,Asian equities traded in the US as American de...,Wipro Ltd.
466,"(Corrects to ""executive"" from ""CTO"" in headlin...",Wipro Ltd.
467,Wipro Ltd (NYSE:WIT) is one of Goldman Sachs’ ...,Wipro Ltd.
468,Asian equities traded in the US as American de...,Wipro Ltd.


In [82]:
stock_analytics.to_csv("D:/Stock Analyzer/stock_analytics.csv",index=False)
stock_news.to_csv("D:/Stock Analyzer/stock_news.csv",index=False)